# Lecture 06 - Professor Solution Notebook

Updated integrative case using Meta Platforms (`META`) adjusted close prices from 2018 through July 10, 2026.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv("Lecture 06 - meta daily 2018 2026.csv")
data["Date"] = pd.to_datetime(data["Date"])
data = data.sort_values("Date")
data["simple_return"] = data["Adj Close"].pct_change()
returns = data.dropna(subset=["simple_return"]).copy()
r = returns["simple_return"]
returns.head()

In [ ]:
setup = pd.Series({
    "asset": "Meta Platforms",
    "ticker": "META",
    "price_observations": len(data),
    "return_observations": len(returns),
    "first_return_date": returns["Date"].iloc[0].date(),
    "last_return_date": returns["Date"].iloc[-1].date(),
    "price_column": "Adj Close",
    "return_type": "simple daily return",
})
setup

In [ ]:
summary = pd.Series({
    "count": r.count(),
    "mean": r.mean(),
    "median": r.median(),
    "standard_deviation": r.std(ddof=1),
    "standard_error_mean": r.std(ddof=1) / np.sqrt(r.count()),
    "minimum": r.min(),
    "p01": r.quantile(0.01),
    "p05": r.quantile(0.05),
    "p95": r.quantile(0.95),
    "p99": r.quantile(0.99),
    "maximum": r.max(),
    "share_negative_days": (r < 0).mean(),
    "skewness": r.skew(),
    "excess_kurtosis": r.kurt(),
})
summary

In [ ]:
annualized_volatility = r.std(ddof=1) * np.sqrt(252)
annualized_mean = (1 + r.mean()) ** 252 - 1
annualized_volatility, annualized_mean

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(data["Date"], data["Adj Close"])
plt.title("META adjusted close price")
plt.xlabel("Date")
plt.ylabel("Adjusted close")
plt.show()

In [ ]:
mu = r.mean()
sigma = r.std(ddof=1)
x = np.linspace(r.quantile(0.002), r.quantile(0.998), 400)
normal_pdf = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

plt.figure(figsize=(10, 5))
plt.hist(r, bins=60, density=True, alpha=0.7, label="Empirical returns")
plt.plot(x, normal_pdf, linewidth=2, label="Normal benchmark")
plt.title("META daily returns: empirical histogram and normal benchmark")
plt.xlabel("Daily simple return")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
normal_p05 = mu + (-1.645) * sigma
tail_comparison = pd.Series({
    "empirical_p05": r.quantile(0.05),
    "normal_p05": normal_p05,
    "empirical_below_minus_2sigma": (r < mu - 2 * sigma).mean(),
    "normal_below_minus_2sigma": 0.0228,
    "empirical_below_minus_3sigma": (r < mu - 3 * sigma).mean(),
    "normal_below_minus_3sigma": 0.00135,
})
tail_comparison

In [ ]:
returns["loss_today"] = returns["simple_return"] < 0
returns["loss_yesterday"] = returns["loss_today"].shift(1)
loss_lagged = returns.dropna(subset=["loss_yesterday"]).copy()
loss_lagged["loss_yesterday"] = loss_lagged["loss_yesterday"].astype(bool)

loss_probs = pd.Series({
    "P(loss)": loss_lagged["loss_today"].mean(),
    "P(loss | previous loss)": loss_lagged.loc[loss_lagged["loss_yesterday"], "loss_today"].mean(),
    "P(loss | previous non-loss)": loss_lagged.loc[~loss_lagged["loss_yesterday"], "loss_today"].mean(),
})
loss_table = pd.crosstab(loss_lagged["loss_yesterday"], loss_lagged["loss_today"])
loss_probs, loss_table

In [ ]:
large_loss_threshold = r.quantile(0.05)
returns["large_loss_today"] = returns["simple_return"] <= large_loss_threshold
returns["large_loss_yesterday"] = returns["large_loss_today"].shift(1)
large_lagged = returns.dropna(subset=["large_loss_yesterday"]).copy()
large_lagged["large_loss_yesterday"] = large_lagged["large_loss_yesterday"].astype(bool)

large_loss_probs = pd.Series({
    "large_loss_threshold": large_loss_threshold,
    "P(large loss)": large_lagged["large_loss_today"].mean(),
    "P(large loss | previous large loss)": large_lagged.loc[large_lagged["large_loss_yesterday"], "large_loss_today"].mean(),
    "P(large loss | not previous large loss)": large_lagged.loc[~large_lagged["large_loss_yesterday"], "large_loss_today"].mean(),
    "previous_large_loss_days": large_lagged["large_loss_yesterday"].sum(),
    "consecutive_large_loss_days": (large_lagged["large_loss_yesterday"] & large_lagged["large_loss_today"]).sum(),
})
large_loss_table = pd.crosstab(large_lagged["large_loss_yesterday"], large_lagged["large_loss_today"])
large_loss_probs, large_loss_table

## Interpretation

Ordinary loss days do not show strong one-day clustering in this sample. Large-loss days are more likely after a previous large-loss day, but the number of consecutive large-loss cases is small, so the interpretation should remain cautious.